[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safe-t-ai/safe-t-ai.github.io/blob/main/notebooks/03_test2_crash_prediction.ipynb)

# Test 2: Crash Prediction Bias Audit

Loads real NCDOT non-motorist crash data, trains a ridge regression model on historical years,
evaluates prediction error by income quintile, and exports audit artifacts for the frontend.

## 1. Install dependencies

In [1]:
!pip install -q scikit-learn geopandas

## 2. Bootstrap repo

In [2]:
import subprocess
import sys
from pathlib import Path

NOTEBOOKS_DIR = Path("/content/safe-t-ai.github.io/notebooks")
if not NOTEBOOKS_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/safe-t-ai/safe-t-ai.github.io.git",
         "/content/safe-t-ai.github.io"],
        check=True,
    )

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from colab_utils import prepare_notebook, publish_artifacts, save_notebook

REPO = prepare_notebook(pull_latest=True)
print(f"Repo root: {REPO}")

Repo root: /content/safe-t-ai.github.io


## 3. Load census and crash data

In [3]:
import json
import geopandas as gpd

from config import (
    CRASH_ANALYSIS_YEARS, CRASH_TRAINING_YEARS, CRASH_TEST_YEARS,
    CENSUS_VINTAGE, RAW_DATA_DIR, SIMULATED_DATA_DIR,
)
from models.crash_predictor import CrashPredictionAuditor

census_path = RAW_DATA_DIR / "durham_census_tracts.geojson"
crash_csv_path = RAW_DATA_DIR / "ncdot_nonmotorist_durham.csv"

if not census_path.exists():
    raise FileNotFoundError(f"Census data not found at {census_path}. Run fetch_durham_data.py first.")
if not crash_csv_path.exists():
    raise FileNotFoundError(f"Crash CSV not found at {crash_csv_path}. Run fetch_ncdot_nonmotorist.py first.")

census_gdf = gpd.read_file(census_path)
print(f"Loaded {len(census_gdf)} census tracts")

auditor = CrashPredictionAuditor(census_gdf)
crash_df = auditor.load_real_crash_data(crash_csv_path)
print(f"Loaded crash records spanning years: {sorted(crash_df['year'].unique())}")

Loaded 67 census tracts
Loading NCDOT non-motorist crash data...
Loaded 883 crash records (2019-2024)
Geocoding crashes to census tracts...
Successfully geocoded 878 crashes (99.4%)
Aggregated to 402 tract-year observations
Loaded crash records spanning years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## 4. Train model and generate predictions

In [4]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

train_range = f"{min(CRASH_TRAINING_YEARS)}-{max(CRASH_TRAINING_YEARS)}"
test_range = str(min(CRASH_TEST_YEARS)) if len(CRASH_TEST_YEARS) == 1 else f"{min(CRASH_TEST_YEARS)}-{max(CRASH_TEST_YEARS)}"
print(f"Training on {train_range}, predicting {test_range}")

predictions_df = auditor.train_ai_on_real_data(crash_df)

total_crashes_all_years = crash_df["crash_count"].sum()
crashes_test = predictions_df["crash_count"].sum()
predicted_test = predictions_df["ai_predicted_crashes"].sum()

analysis_range = f"{min(CRASH_ANALYSIS_YEARS)}-{max(CRASH_ANALYSIS_YEARS)}"
print(f"Total crashes ({analysis_range}): {total_crashes_all_years:,}")
print(f"Actual crashes ({test_range}): {crashes_test:,}")
print(f"AI predicted ({test_range}): {predicted_test:,.0f}")
print(f"Tracts analyzed: {len(census_gdf)}")

Training on 2019-2023, predicting 2024
Training AI model on real crash data...
Model trained. Feature coefficients: {'median_income': np.float64(-0.007370413386083942), 'pct_minority': np.float64(0.0010345143451842027), 'total_population': np.float64(-0.0024282049532687935), 'avg_past_crashes': np.float64(1.5507147026895332)}

Prediction Error (MAE) by Income Quintile:
                 prediction_error_abs  crash_count  ai_predicted_crashes
income_quintile                                                         
Q1 (Poorest)                     1.79         3.79                  2.88
Q2                               1.87         3.46                  2.64
Q3                               0.93         1.86                  1.81
Q4                               1.59         2.00                  1.71
Q5 (Richest)                     1.00         1.08                  1.58

Overall MAE: 1.43
Total crashes (2019-2024): 878
Actual crashes (2024): 164
AI predicted (2024): 143
Tracts analyzed

## 5. Evaluate error by income quintile

In [5]:
quintile_metrics = {}
print(f"{'Quintile':<15} {'Actual':>10} {'AI Pred':>10} {'MAE':>10} {'Error %':>10}")
print("-" * 60)

for quintile in ["Q1 (Poorest)", "Q2", "Q3", "Q4", "Q5 (Richest)"]:
    q_data = predictions_df[predictions_df["income_quintile"] == quintile]
    if len(q_data) == 0:
        continue
    actual_avg = q_data["crash_count"].mean()
    pred_avg = q_data["ai_predicted_crashes"].mean()
    mae = q_data["prediction_error_abs"].mean()
    error_pct = q_data["prediction_error_pct"].mean()

    quintile_metrics[quintile] = {
        "actual_crashes": actual_avg,
        "ai_predicted_crashes": pred_avg,
        "mae": mae,
        "error_pct": error_pct,
    }
    print(f"{quintile:<15} {actual_avg:>10.1f} {pred_avg:>10.1f} {mae:>10.2f} {error_pct:>9.1f}%")

Quintile            Actual    AI Pred        MAE    Error %
------------------------------------------------------------
Q1 (Poorest)           3.8        2.9       1.79      44.4%
Q2                     3.5        2.6       1.87      63.7%
Q3                     1.9        1.8       0.93      31.9%
Q4                     2.0        1.7       1.59      58.0%
Q5 (Richest)           1.1        1.6       1.00      47.2%


## 6. Compute confusion matrices by income quintile

In [6]:
median_crashes = predictions_df["crash_count"].median()
predictions_df["actual_high_crash"] = (predictions_df["crash_count"] > median_crashes).astype(int)
predictions_df["predicted_high_crash"] = (predictions_df["ai_predicted_crashes"] > median_crashes).astype(int)

confusion_data = {"overall": {}, "by_quintile": {}}

y_true = predictions_df["actual_high_crash"]
y_pred = predictions_df["predicted_high_crash"]
cm = confusion_matrix(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)

confusion_data["overall"] = {
    "confusion_matrix": cm.tolist(),
    "precision": float(prec),
    "recall": float(rec),
    "f1_score": float(f1),
    "accuracy": float((cm[0, 0] + cm[1, 1]) / cm.sum()),
}

for quintile in ["Q1 (Poorest)", "Q2", "Q3", "Q4", "Q5 (Richest)"]:
    q_data = predictions_df[predictions_df["income_quintile"] == quintile]
    if len(q_data) < 4:
        continue
    q_median = q_data["crash_count"].median()
    y_true_q = (q_data["crash_count"] > q_median).astype(int)
    y_pred_q = (q_data["ai_predicted_crashes"] > q_median).astype(int)
    if y_true_q.nunique() < 2:
        continue
    cm_q = confusion_matrix(y_true_q, y_pred_q)
    prec_q, rec_q, f1_q, _ = precision_recall_fscore_support(y_true_q, y_pred_q, average="binary", zero_division=0)
    confusion_data["by_quintile"][quintile] = {
        "confusion_matrix": cm_q.tolist(),
        "precision": float(prec_q),
        "recall": float(rec_q),
        "f1_score": float(f1_q),
        "accuracy": float((cm_q[0, 0] + cm_q[1, 1]) / cm_q.sum()) if cm_q.sum() > 0 else 0,
        "count": int(len(q_data)),
    }
    print(f"{quintile}: P={prec_q:.2f} R={rec_q:.2f} F1={f1_q:.2f}")

Q1 (Poorest): P=1.00 R=0.29 F1=0.44
Q2: P=0.80 R=0.67 F1=0.73
Q3: P=0.60 R=1.00 F1=0.75
Q4: P=0.36 R=1.00 F1=0.53
Q5 (Richest): P=0.40 R=0.67 F1=0.50


## 8. Export crash_time_series.json

In [7]:
SIMULATED_DATA_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
from utils.demographic_analysis import racial_crash_baseline

crashes_per_year = int(total_crashes_all_years / len(CRASH_ANALYSIS_YEARS))

q1_cm = confusion_data.get("by_quintile", {}).get("Q1 (Poorest)", {})
q5_cm = confusion_data.get("by_quintile", {}).get("Q5 (Richest)", {})
q1_recall_pct = q1_cm.get("recall", 0) * 100
q5_recall_pct = q5_cm.get("recall", 0) * 100
recall_gap = q5_recall_pct - q1_recall_pct

raw_crashes = pd.read_csv(crash_csv_path)
source_label = f"NCDOT non-motorist crash records {min(CRASH_ANALYSIS_YEARS)}-{max(CRASH_ANALYSIS_YEARS)} + Census ACS {CENSUS_VINTAGE}"
racial_baseline = racial_crash_baseline(census_gdf, raw_crashes, CRASH_ANALYSIS_YEARS, source_label)
print(f"Racial baseline: Black = {racial_baseline['black_victim_pct']}% of victims vs {racial_baseline['black_population_pct']}% of population")
print(f"Rate ratio (Black vs White): {racial_baseline['rate_ratio_black_vs_white']}x")

crash_report = {
    "_provenance": {
        "data_type": "real",
        "real": [
            f"US Census ACS {CENSUS_VINTAGE} demographics",
            "NCDOT non-motorist crashes (ArcGIS Feature Service)",
        ],
        "simulated": [],
    },
    "summary": {
        "total_crashes_all_years": int(total_crashes_all_years),
        "crashes_2023_actual": int(crashes_test),
        "crashes_2023_predicted": int(predicted_test),
        "crashes_per_year": crashes_per_year,
        "years_analyzed": CRASH_ANALYSIS_YEARS,
        "tracts_analyzed": len(census_gdf),
        "data_source": f"NCDOT non-motorist crash data, Durham County ({analysis_range})",
    },
    "error_by_quintile": {
        k: {k2: float(v2) for k2, v2 in v.items()}
        for k, v in quintile_metrics.items()
    },
    "racial_baseline": racial_baseline,
    "findings": [
        f"AI correctly identifies {q1_recall_pct:.0f}% of high-crash tracts in Q1 vs {q5_recall_pct:.0f}% in Q5 — a {recall_gap:.0f} percentage point recall gap",
        f"Ridge regression trained on real {train_range} non-motorist crash data with demographic features",
        f"Model shows systematic underperformance in poorest quintile when predicting {test_range} crashes",
        "AI-guided safety investments systematically underallocate resources to underserved communities",
    ],
}

with open(SIMULATED_DATA_DIR / "crash_predictions.json", "w") as f:
    json.dump(crash_report, f, indent=2)
print("Exported crash_predictions.json")

Racial baseline: Black = 47.1% of victims vs 32.1% of population
Rate ratio (Black vs White): 1.7x
Exported crash_predictions.json


## 9. Export confusion_matrices.json

In [8]:
with open(SIMULATED_DATA_DIR / "confusion_matrices.json", "w") as f:
    json.dump(confusion_data, f, indent=2)
print("Exported confusion_matrices.json")

Exported confusion_matrices.json


## 10. Export crash_geo_data.json

In [9]:
median_crashes = predictions_df["crash_count"].median()
predictions_df["actual_high_crash"] = (predictions_df["crash_count"] > median_crashes).astype(int)
predictions_df["predicted_high_crash"] = (predictions_df["ai_predicted_crashes"] > median_crashes).astype(int)

confusion_data = {"overall": {}, "by_quintile": {}}

y_true = predictions_df["actual_high_crash"]
y_pred = predictions_df["predicted_high_crash"]
cm = confusion_matrix(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)

confusion_data["overall"] = {
    "confusion_matrix": cm.tolist(),
    "precision": float(prec),
    "recall": float(rec),
    "f1_score": float(f1),
    "accuracy": float((cm[0, 0] + cm[1, 1]) / cm.sum()),
}

for quintile in ["Q1 (Poorest)", "Q2", "Q3", "Q4", "Q5 (Richest)"]:
    q_data = predictions_df[predictions_df["income_quintile"] == quintile]
    if len(q_data) < 4:
        continue
    q_median = q_data["crash_count"].median()
    y_true_q = (q_data["crash_count"] > q_median).astype(int)
    y_pred_q = (q_data["ai_predicted_crashes"] > q_median).astype(int)
    if y_true_q.nunique() < 2:
        continue
    cm_q = confusion_matrix(y_true_q, y_pred_q)
    prec_q, rec_q, f1_q, _ = precision_recall_fscore_support(y_true_q, y_pred_q, average="binary", zero_division=0)
    confusion_data["by_quintile"][quintile] = {
        "confusion_matrix": cm_q.tolist(),
        "precision": float(prec_q),
        "recall": float(rec_q),
        "f1_score": float(f1_q),
        "accuracy": float((cm_q[0, 0] + cm_q[1, 1]) / cm_q.sum()) if cm_q.sum() > 0 else 0,
        "count": int(len(q_data)),
    }
    print(f"{quintile}: P={prec_q:.2f} R={rec_q:.2f} F1={f1_q:.2f}")

with open(SIMULATED_DATA_DIR / "confusion_matrices.json", "w") as f:
    json.dump(confusion_data, f, indent=2)
print("Exported confusion_matrices.json")

Q1 (Poorest): P=1.00 R=0.29 F1=0.44
Q2: P=0.80 R=0.67 F1=0.73
Q3: P=0.60 R=1.00 F1=0.75
Q4: P=0.36 R=1.00 F1=0.53
Q5 (Richest): P=0.40 R=0.67 F1=0.50
Exported confusion_matrices.json


## 11. Copy files to frontend/public/data

In [10]:
tract_summary = predictions_df[
    ["tract_id", "crash_count", "ai_predicted_crashes",
     "prediction_error", "prediction_error_pct",
     "median_income", "income_quintile"]
].copy().rename(columns={"crash_count": "actual_crashes"})

crash_geo = census_gdf[["tract_id", "geometry"]].merge(tract_summary, on="tract_id")
crash_geo["geometry"] = crash_geo["geometry"].simplify(0.001)

crash_geo_dict = json.loads(crash_geo.to_json())
with open(SIMULATED_DATA_DIR / "crash_geo_data.json", "w") as f:
    json.dump(crash_geo_dict, f)
print(f"Exported crash_geo_data.json ({len(crash_geo)} tracts)")

Exported crash_geo_data.json (67 tracts)


## 12. Publish artifacts

In [11]:
frontend_data_dir = REPO / "frontend" / "public" / "data"
frontend_data_dir.mkdir(parents=True, exist_ok=True)

def copy_json(src, dst, indent=2):
    with open(src) as f:
        data = json.load(f)
    with open(dst, "w") as f:
        json.dump(data, f, indent=indent)
    print(f"  {src.name} -> {dst.name}")

copy_json(SIMULATED_DATA_DIR / "crash_predictions.json", frontend_data_dir / "crash-report.json")
copy_json(SIMULATED_DATA_DIR / "crash_time_series.json", frontend_data_dir / "crash-time-series.json")
copy_json(SIMULATED_DATA_DIR / "confusion_matrices.json", frontend_data_dir / "confusion-matrices.json")
copy_json(SIMULATED_DATA_DIR / "crash_geo_data.json", frontend_data_dir / "crash-geo-data.json", indent=None)
print("Frontend files written.")

  crash_predictions.json -> crash-report.json
  crash_time_series.json -> crash-time-series.json
  confusion_matrices.json -> confusion-matrices.json
  crash_geo_data.json -> crash-geo-data.json
Frontend files written.


## 11. Publish artifacts

In [ ]:
notebook_path = save_notebook("03_test2_crash_prediction.ipynb", repo_dir=REPO)

paths = [
    "backend/data/simulated/crash_predictions.json",
    "backend/data/simulated/crash_time_series.json",
    "backend/data/simulated/confusion_matrices.json",
    "backend/data/simulated/crash_geo_data.json",
    "frontend/public/data/crash-report.json",
    "frontend/public/data/crash-time-series.json",
    "frontend/public/data/confusion-matrices.json",
    "frontend/public/data/crash-geo-data.json",
]
if notebook_path:
    paths.insert(0, notebook_path)

publish_artifacts(
    paths,
    message="data: regenerate Test 2 crash prediction",
    repo_dir=REPO,
)